<style>
.topic-header { background: linear-gradient(135deg, #e8f4f8 0%, #d4e8f0 100%); border-left: 4px solid #5ba4c9; padding: 16px 20px; border-radius: 0 8px 8px 0; margin: 10px 0; font-size: 15px; color: #1a3a4a; }
.concept-box { background: #eef6fa; border: 1px solid #c4dce8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a4a5a; }
.try-it { background: #fef9e7; border: 1px solid #f0d87a; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #5a4a1a; }
.takeaway { background: #e8f5e8; border: 1px solid #a8d5a8; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #2a5a2a; }
.warning-box { background: #fdf0f0; border: 1px solid #e8b0b0; border-radius: 8px; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #6a2a2a; }
.where-box { background: #fff3e0; border-left: 4px solid #ff9800; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #4a3000; }
.fix-box { background: #e8f5e9; border-left: 4px solid #4caf50; border-radius: 0 8px 8px 0; padding: 14px 18px; margin: 8px 0; font-size: 14px; color: #1b5e20; }
.diagram-box { background: #f8f9fa; border: 2px solid #dee2e6; border-radius: 12px; padding: 24px; margin: 16px 0; text-align: center; }
.compare-table { width: 100%; border-collapse: collapse; margin: 12px 0; }
.compare-table th { background: #d4e8f0; color: #1a3a4a; padding: 10px 14px; text-align: left; border: 1px solid #c4dce8; }
.compare-table td { padding: 10px 14px; border: 1px solid #dee2e6; font-size: 13px; }
.compare-table tr:nth-child(even) { background: #f8fbfd; }
.section-divider { border: none; border-top: 2px solid #d4e8f0; margin: 25px 0; }
</style>

<div class='topic-header'>
<h1>E08 &middot; Day 3 &middot; RAG End-to-End &mdash; Ground the Model in YOUR Documents</h1>
<p><strong>GenAI for Engineering Managers &bull; Exercise 8 of 15 &bull; Opens Day 3</strong> &nbsp;|&nbsp; From an assistant that chats fluently to one that answers from your handbook and runbooks &mdash; with sources</p>
</div>

**Why this matters at your altitude.** Every "AI assistant for our docs" pitch your teams will bring you &mdash; support copilots, onboarding bots, runbook assistants &mdash; is, underneath, the pattern in this notebook: **Retrieval-Augmented Generation (RAG)**. In the next hour you will watch a raw model fail on questions about our own engineering handbook, then watch the *same model* answer them correctly &mdash; with citations &mdash; after we give it a retrieval layer. No fine-tuning, no training run, no data leaving the room. When a vendor quotes months for "training the AI on your documents", this session is your calibration for what that sentence should actually mean, cost, and take.

<div style="background: linear-gradient(135deg, #e8f0f8 0%, #d0dce8 100%); padding: 30px 32px; border-radius: 12px; margin-bottom: 20px;">
<h1 style="color: #1a2a4a; margin: 0; font-size: 28px;">A3 &middot; Day 4 &middot; The Incident Triage Agent &mdash; With a Human Gate</h1>
<p style="color: #3a6a8a; margin: 8px 0 0 0; font-size: 16px;">GenAI for Engineering Managers &mdash; Exercise 3 of 3 &middot; Case study, ending on a working interface</p>
<p style="color: #2a4a5a; margin: 14px 0 0 0; font-size: 14px; line-height: 1.6;">
<strong>The gap A2 left:</strong> our agent can act, remember across restarts, and check the outside
world. Nobody has approved any of it. In A1 it changed our system of record simply because a tool
existed.<br><br>
This session puts a <strong>human in front of the action</strong> &mdash; first in code, then behind a
real Approve / Reject interface &mdash; and proves, with file fingerprints, that a rejected action
leaves the system of record untouched.
</p>
</div>

<div class='concept-box'>
<strong>It is 13:10.</strong> The store-systems incident queue holds five open tickets and one that
arrives later. Triage is always the same four moves: <strong>classify severity, consult the runbook,
check the live systems, act.</strong><br><br>
Today the agent makes the first three moves alone and <strong>stops dead before the fourth</strong> &mdash;
because acting changes state, and state changes go past a human.
</div>

<div class='concept-box'>
<strong>The architecture &mdash; three words:</strong><br><br>
<b>KNOW</b> &mdash; <code>lookup_runbook</code>: the written procedure. Static, so it is RAG-shaped.<br>
<b>NOW</b> &mdash; <code>check_system_status</code>: live telemetry. Changes minute to minute, so it is
tool-shaped.<br>
<b>DO</b> &mdash; <code>propose_action</code>: the only path to a state change, and it
<strong>cannot execute</strong>. It stages.
</div>

<hr class='section-divider'>

## Part 1 &mdash; Setup and the Queue

In [ ]:
%pip install -qU openai docx2txt gradio

In [ ]:
import os, json, hashlib, shutil, datetime

os.environ['OPENAI_API_KEY'] = 'PASTE_THE_KEY_SHARED_IN_SESSION_HERE'

from openai import OpenAI
client = OpenAI()
MODEL  = 'gpt-4.1-nano'

TICKETS_FILE = '../data/incident_tickets.json'
AUDIT = []            # everything that actually happened, in order

# Snapshot so the notebook can be re-run from a clean state.
shutil.copy(TICKETS_FILE, '../data/_a3_backup.json')

def fingerprint():
    return hashlib.sha256(open(TICKETS_FILE, 'rb').read()).hexdigest()[:16]

def load_tickets():
    return json.load(open(TICKETS_FILE))['tickets']

print('Model:', MODEL)
print('Queue fingerprint at start:', fingerprint())

In [ ]:
%run ../data/setup_incident_queue.py

for t in load_tickets():
    print(f"{t['id']}  [{t['status']:>8}]  {t['store']:<34} {t['title']}")

<hr class='section-divider'>

## Part 2 &mdash; KNOW: the runbook

<div class='concept-box'>
The procedure your team already wrote down. Nothing about it changes minute to minute, so it is the
static half &mdash; exactly the material RAG was built for in Day 3. We read it from the same
<code>.docx</code> we indexed in E08.
</div>

In [ ]:
import docx2txt
%run ../data/setup_e08_docs.py

RUNBOOK_TEXT = docx2txt.process('../data/store_systems_runbook.docx')

def lookup_runbook(alert):
    """KNOW — return the written procedure relevant to an alert."""
    lines = [l.strip() for l in RUNBOOK_TEXT.splitlines() if l.strip()]
    key   = alert.upper()
    hits  = [l for l in lines if key.split('_')[0] in l.upper() or key in l.upper()]
    if not hits:
        hits = lines[:12]
    return {'alert': alert, 'procedure': ' | '.join(hits[:12])[:1200]}

print(json.dumps(lookup_runbook('POS_SYNC_LAG'), indent=2)[:700])

<hr class='section-divider'>

## Part 3 &mdash; NOW: live telemetry

<div class='concept-box'>
Simulated here, but these are <em>real functions</em> &mdash; in production they are your observability
APIs. The point is the shape: the runbook tells you the threshold, telemetry tells you the current
value, and neither is useful without the other.
</div>

In [ ]:
TELEMETRY = {
    'INC-30017': {'store': 2203, 'card_auth_success_rate': 0.04, 'acquirer_response_ms': 31000,
                  'cash_transactions': 'normal', 'affected_registers': 22, 'total_registers': 24},
    'INC-30021': {'store': 4471, 'pos_sync_backlog': 1180, 'gateway_heartbeat': 'OK',
                  'broker_partition_lag': 142000, 'region_peers_affected': 0},
    'INC-30024': {'stores_affected': 14, 'feed_stale_minutes': 62, 'pickup_orders_accepted': 431,
                  'region': 'South-Central'},
    'INC-30028': {'store': 554, 'median_checkout_seconds': 15.2, 'payment_token_cache': 'EXPIRED',
                  'network_loss_pct': 0.0},
    'INC-30031': {'store': 3110, 'note': 'no alert has fired for this store',
                  'pos_sync_backlog': 12, 'median_checkout_seconds': 2.1, 'gateway_heartbeat': 'OK'},
    'INC-30047': {'store': 1188, 'lane': 4, 'coupon_engine_version': '7.2.1',
                  'other_lanes_version': '7.2.4', 'coupon_rejections_last_hour': 38},
}

def check_system_status(ticket_id):
    """NOW — current live readings for the store on this ticket."""
    return TELEMETRY.get(ticket_id.upper(), {'error': f'no telemetry for {ticket_id}'})

print(json.dumps(check_system_status('INC-30021'), indent=2))

<hr class='section-divider'>

## Part 4 &mdash; DO: the tool that cannot do anything

<div class='concept-box'>
This is the whole design. The agent gets <strong>one</strong> write-shaped tool, and that tool
<em>stages</em> a proposal. It has no code path that touches the ticket file.<br><br>
The function that <em>does</em> write, <code>execute_action</code>, is <strong>not in the agent's
toolbox at all</strong>. Only a human approval can reach it. That is not a policy or a prompt
instruction &mdash; it is a structural fact about the code, which is why it holds even if the model
misbehaves.
</div>

In [ ]:
PENDING = {}          # ticket_id -> staged proposal, awaiting a human

def propose_action(ticket_id, severity, new_status, action, reason):
    """DO (staged) — the ONLY write-shaped tool the agent can see. It executes nothing."""
    proposal = {'ticket_id': ticket_id, 'severity': severity, 'new_status': new_status,
                'action': action, 'reason': reason,
                'staged_at': datetime.datetime.now().isoformat(timespec='seconds')}
    PENDING[ticket_id.upper()] = proposal
    return {'staged': True, 'awaiting_human_approval': True, **proposal}


def execute_action(ticket_id):
    """NOT a tool. Only a human approval reaches this."""
    p = PENDING.get(ticket_id.upper())
    if not p:
        return {'ok': False, 'error': 'nothing staged'}
    data = json.load(open(TICKETS_FILE))
    for t in data['tickets']:
        if t['id'].upper() == ticket_id.upper():
            t['status']            = p['new_status']
            t['severity_reported'] = p['severity']
            t.setdefault('audit', []).append({'action': p['action'], 'by': 'agent, human-approved',
                                              'at': datetime.datetime.now().isoformat(timespec='seconds')})
            json.dump(data, open(TICKETS_FILE, 'w'), indent=2)
            AUDIT.append(('EXECUTED', ticket_id, p['action']))
            del PENDING[ticket_id.upper()]
            return {'ok': True, **p}
    return {'ok': False, 'error': 'no such ticket'}


def reject_action(ticket_id, why='rejected by on-call'):
    """Discard a staged proposal. Touches nothing."""
    p = PENDING.pop(ticket_id.upper(), None)
    AUDIT.append(('REJECTED', ticket_id, why))
    return {'ok': True, 'discarded': p is not None}

print('Agent-visible write tools :', ['propose_action'])
print('Human-only functions      :', ['execute_action', 'reject_action'])

In [ ]:
TOOL_SPECS = [
    {'type': 'function', 'function': {
        'name': 'lookup_runbook',
        'description': 'KNOW: read the written procedure for an alert, including its thresholds.',
        'parameters': {'type': 'object', 'properties': {'alert': {'type': 'string'}},
                       'required': ['alert']}}},
    {'type': 'function', 'function': {
        'name': 'check_system_status',
        'description': 'NOW: current live telemetry for the store on this ticket.',
        'parameters': {'type': 'object', 'properties': {'ticket_id': {'type': 'string'}},
                       'required': ['ticket_id']}}},
    {'type': 'function', 'function': {
        'name': 'propose_action',
        'description': 'Stage a triage decision for human approval. This does NOT execute anything.',
        'parameters': {'type': 'object', 'properties': {
            'ticket_id':  {'type': 'string'},
            'severity':   {'type': 'string', 'description': 'SEV-1, SEV-2, SEV-3 or UNCLEAR'},
            'new_status': {'type': 'string'},
            'action':     {'type': 'string', 'description': 'the single next action, in one sentence'},
            'reason':     {'type': 'string', 'description': 'cite the runbook threshold and the live reading'}},
            'required': ['ticket_id', 'severity', 'new_status', 'action', 'reason']}}},
]

TOOL_FUNCS = {'lookup_runbook': lookup_runbook,
              'check_system_status': check_system_status,
              'propose_action': propose_action}

SYSTEM = (
    'You are the store-systems triage agent. For the ticket you are given: '
    '(1) consult the runbook, (2) check live telemetry, (3) call propose_action exactly once. '
    'Cite the runbook threshold and the live reading in your reason. '
    'You operate at autonomy dial setting 2: PROPOSE ONLY. '
    'You must never claim to have changed, escalated or fixed anything — you can only propose. '
    'If the evidence is too thin to classify severity, propose severity UNCLEAR and say what is missing.'
)

def triage(ticket_id, show_trace=True):
    t = next(x for x in load_tickets() if x['id'] == ticket_id)
    messages = [{'role': 'system', 'content': SYSTEM},
                {'role': 'user', 'content': f"Ticket {t['id']} — {t['title']}\nStore: {t['store']}\n"
                                            f"Reported by: {t['reported_by']}\n\n{t['body']}"}]
    trace = []
    for step in range(8):
        msg = client.chat.completions.create(model=MODEL, messages=messages,
                                             tools=TOOL_SPECS).choices[0].message
        if not msg.tool_calls:
            return msg.content.strip(), trace
        messages.append(msg)
        for call in msg.tool_calls:
            args = json.loads(call.function.arguments or '{}')
            trace.append(call.function.name)
            if show_trace:
                print(f'   [step {step + 1}] {call.function.name}')
            messages.append({'role': 'tool', 'tool_call_id': call.id,
                             'content': json.dumps(TOOL_FUNCS[call.function.name](**args))[:2500]})
    return '(too many steps)', trace

print('Triage agent ready.')

<hr class='section-divider'>

## Part 5 &mdash; Triage the Clearest Case, and Prove It Stopped

<div class='concept-box'>
INC-30021: the POS sync backlog. Watch the trace &mdash; runbook, then telemetry, then a proposal &mdash;
and watch the fingerprint.
</div>

In [ ]:
before = fingerprint()

print('Q: triage INC-30021')
print('  -- trace --')
summary, trace = triage('INC-30021')
print('\nAGENT SAYS:', summary[:300])

print('\nSTAGED PROPOSAL:')
print(json.dumps(PENDING.get('INC-30021'), indent=2))

print('\nTHE FILE:')
print(f'  fingerprint before : {before}')
print(f'  fingerprint after  : {fingerprint()}')
print(f'  INC-30021 status   : {next(t["status"] for t in load_tickets() if t["id"] == "INC-30021")}')
print(f'\n  FILE ACTUALLY CHANGED: {before != fingerprint()}')

<div class='takeaway'>
<strong>It did the work and then stopped.</strong> It read the runbook, read the telemetry, weighed the
backlog against the documented thresholds, and wrote a proposal with its reasoning &mdash; and the
system of record is byte-identical.<br><br>
Compare this with A1 Part 6, where the agent changed the file the moment a write tool existed. The
difference is not a better prompt. <strong>The write function is not in the toolbox.</strong>
</div>

<hr class='section-divider'>

## Part 6 &mdash; The Gate: Approve One, Reject One

<div class='concept-box'>
Two staged proposals, two human decisions, and a fingerprint check after each. This is the part to
run slowly.
</div>

In [ ]:
print('DECISION 1 — APPROVE INC-30021')
print('=' * 72)
before = fingerprint()
result = execute_action('INC-30021')
print('  result             :', result['ok'])
print(f'  fingerprint        : {before}  ->  {fingerprint()}')
print(f'  INC-30021 status   : {next(t["status"] for t in load_tickets() if t["id"] == "INC-30021")}')
print(f'  FILE CHANGED       : {before != fingerprint()}')

In [ ]:
print('DECISION 2 — REJECT INC-30024')
print('=' * 72)
print('  -- trace --')
triage('INC-30024')
print('\n  staged:', PENDING.get('INC-30024', {}).get('action', '(none)')[:90])

before = fingerprint()
reject_action('INC-30024', why='want a human to confirm pickup-order impact first')

print(f'\n  fingerprint        : {before}  ->  {fingerprint()}')
print(f'  INC-30024 status   : {next(t["status"] for t in load_tickets() if t["id"] == "INC-30024")}')
print(f'  still pending?     : {"INC-30024" in PENDING}')
print(f'  FILE CHANGED       : {before != fingerprint()}')

<div class='takeaway'>
<strong>Approve changed the file. Reject changed nothing.</strong> Not "logged a rejection and carried
on" &mdash; the fingerprint is identical and the proposal is gone.<br><br>
That is the property to demand in a design review: <strong>show me the rejection path, and show me the
hash.</strong> Plenty of systems ask for approval and then act anyway on a timeout, a retry, or a
default. Ask which one yours does.
</div>

<hr class='section-divider'>

## Part 7 &mdash; The Same Gate, as an Interface

<div class='concept-box'>
Everything above is the real system. What is missing is the thing an on-call engineer at 2 a.m. can
actually use &mdash; nobody triages an incident by editing a notebook cell.<br><br>
The UI adds <strong>no new capability</strong>. It exposes exactly the functions we already built and
tested: <code>triage</code>, <code>execute_action</code>, <code>reject_action</code>. That is the
correct order to build in &mdash; logic first, proven; interface last.
</div>

In [ ]:
import gradio as gr

def ui_triage(ticket_id):
    if not ticket_id:
        return '_Pick a ticket first._', '', audit_markdown()
    summary, _ = triage(ticket_id, show_trace=False)
    p = PENDING.get(ticket_id.upper())
    if not p:
        return f'**Agent did not stage an action.**\n\n{summary}', '', audit_markdown()
    card = (f"### Proposal for {p['ticket_id']}\n"
            f"| field | value |\n|---|---|\n"
            f"| severity | **{p['severity']}** |\n"
            f"| new status | **{p['new_status']}** |\n"
            f"| action | {p['action']} |\n\n"
            f"**Reasoning:** {p['reason']}\n\n"
            f"_Nothing has been changed. Fingerprint: `{fingerprint()}`_")
    return card, ticket_id, audit_markdown()

def ui_approve(ticket_id):
    if not ticket_id or ticket_id.upper() not in PENDING:
        return '_Nothing staged to approve._', audit_markdown(), fingerprint()
    before = fingerprint()
    r = execute_action(ticket_id)
    return (f"### APPROVED — {ticket_id}\n\n`{before}` &rarr; `{fingerprint()}`\n\n"
            f"**The system of record changed.** New status: **{r['new_status']}**"),\
           audit_markdown(), fingerprint()

def ui_reject(ticket_id):
    if not ticket_id or ticket_id.upper() not in PENDING:
        return '_Nothing staged to reject._', audit_markdown(), fingerprint()
    before = fingerprint()
    reject_action(ticket_id, why='rejected in UI')
    return (f"### REJECTED — {ticket_id}\n\n`{before}` &rarr; `{fingerprint()}`\n\n"
            f"**Unchanged: {before == fingerprint()}.** The proposal was discarded."),\
           audit_markdown(), fingerprint()

def audit_markdown():
    if not AUDIT:
        return '_No decisions yet._'
    rows = '\n'.join(f'| {k} | {t} | {d[:60]} |' for k, t, d in AUDIT)
    return '| decision | ticket | detail |\n|---|---|---|\n' + rows

def queue_markdown():
    rows = '\n'.join(f"| {t['id']} | {t['status']} | {t['title'][:44]} |" for t in load_tickets())
    return '| ticket | status | title |\n|---|---|---|\n' + rows

print('UI handlers defined — same functions as Parts 5 and 6.')

In [ ]:
with gr.Blocks(title='Store Systems — Incident Triage') as triage_ui:
    gr.Markdown('# Store Systems — Incident Triage\n'
                'The agent proposes. **A human decides.** Nothing is written without approval.')
    with gr.Row():
        with gr.Column(scale=1):
            gr.Markdown('### Queue')
            queue_view = gr.Markdown(queue_markdown())
            picker  = gr.Dropdown([t['id'] for t in load_tickets()], label='Ticket')
            run_btn = gr.Button('Run triage', variant='primary')
            fp_view = gr.Textbox(fingerprint(), label='System-of-record fingerprint',
                                 interactive=False)
        with gr.Column(scale=2):
            gr.Markdown('### Agent proposal')
            proposal_view = gr.Markdown('_Pick a ticket and run triage._')
            staged = gr.State('')
            with gr.Row():
                approve_btn = gr.Button('Approve', variant='primary')
                reject_btn  = gr.Button('Reject', variant='stop')
            outcome_view = gr.Markdown()
            gr.Markdown('### Audit log')
            audit_view = gr.Markdown(audit_markdown())

    run_btn.click(ui_triage, picker, [proposal_view, staged, audit_view])
    approve_btn.click(ui_approve, staged, [outcome_view, audit_view, fp_view]) \
               .then(queue_markdown, None, queue_view)
    reject_btn.click(ui_reject, staged, [outcome_view, audit_view, fp_view])

print('UI built. Launch it in the next cell.')

In [ ]:
# Launch. In Jupyter the interface appears inline; the printed URL also works in a browser tab.
# In Colab, add share=True.
triage_ui.launch(prevent_thread_lock=True, quiet=True)
print('Running — open the URL above, triage a ticket, then Approve one and Reject one.')
print('Watch the fingerprint box on the left each time.')

<div class='try-it'>
<strong>Run this live, in this order:</strong>
<ol style='margin:6px 0 0 18px;'>
<li>Triage <strong>INC-30017</strong> (card payments down at lunch rush) &mdash; read the proposed severity.</li>
<li>Press <strong>Reject</strong>. Watch the fingerprint. It does not move.</li>
<li>Triage it again and press <strong>Approve</strong>. Now it moves, and the queue updates.</li>
<li>Triage <strong>INC-30031</strong> &mdash; the "acting weird" ticket with no alert and no detail.</li>
</ol>
<strong>That last one is the honest test.</strong> There is nothing to classify. Watch whether the agent
says so, or invents a severity to look useful.
</div>

<hr class='section-divider'>

## Part 8 &mdash; The Ambiguous Ticket, Audited Honestly

In [ ]:
print('  -- trace --')
summary, _ = triage('INC-30031')
print('\nAGENT SAYS:', summary[:300])
print('\nSTAGED:')
print(json.dumps(PENDING.get('INC-30031'), indent=2))
print('\nWhat telemetry actually shows for this store:')
print(json.dumps(check_system_status('INC-30031'), indent=2))

<div class='try-it'>
<strong>Judge it against the evidence.</strong> The store has no alert firing, a backlog of 12, and normal
checkout times &mdash; the only signal is a second-hand verbal report from someone now off shift.<br><br>
Did the agent mark it <strong>UNCLEAR</strong> and name what is missing, or did it manufacture a severity?
Either outcome is worth the discussion. An agent that always produces a confident answer is not a
capable agent &mdash; it is one that has never been allowed to say "I do not know".
</div>

<div class='warning-box'>
<strong>Facilitator note &mdash; there is a second bug in that answer, and it is worth hunting for.</strong>
In testing, the agent reached the right verdict (UNCLEAR) while <em>misreading the evidence</em>: it
described the backlog of <code>12</code> as "12 minutes" and called it "at the runbook threshold".
It is 12 <em>transactions</em>, and the threshold is 750 &mdash; nowhere near.<br><br>
Exact wording varies run to run, so read what your run produced. If the misreading appears, it is the
best teaching moment in the session: <strong>the conclusion was right and the reasoning was wrong.</strong>
Anyone reviewing only the verdict would have approved it. This is precisely why the proposal card shows
its reasoning rather than just a severity &mdash; and why a human reads that reasoning before pressing
Approve.
</div>

### Exercise &mdash; the sixth ticket is yours

<div class='concept-box'>
<strong>INC-30047</strong> arrived at 13:02: self-checkout lane 4 rejecting all coupons, other lanes fine.
Triage it in the UI. Before you look at the proposal, decide for yourself:<br><br>
1. What severity would <em>you</em> assign, and what in the telemetry justifies it?<br>
2. Does the runbook cover this at all?<br>
3. Would you approve the agent's action, or reject it &mdash; and what would you need to change your mind?
</div>

In [ ]:
print(json.dumps(check_system_status('INC-30047'), indent=2))
print('\n(The version mismatch between lane 4 and the other lanes is the clue.)')
print('\nYour turn — triage INC-30047 in the UI above, then compare with your own answer.')

<hr class='section-divider'>

## Recap &mdash; and the Day

<table class='compare-table'>
<tr><th>Part</th><th>What we proved</th><th>Manager takeaway</th></tr>
<tr><td>4</td><td>The write function is not in the agent's toolbox</td><td>Safety enforced by <strong>structure</strong>, not by prompt instructions</td></tr>
<tr><td>5</td><td>Agent read runbook + telemetry, proposed, file unchanged</td><td>An agent can do the thinking without holding the pen</td></tr>
<tr><td>6</td><td>Approve moved the fingerprint; Reject left it identical</td><td>Ask to see the <em>rejection</em> path, not just the happy path</td></tr>
<tr><td>7</td><td>The UI exposes only functions already proven</td><td>Logic first, interface last — the demo is not the system</td></tr>
<tr><td>8</td><td>A ticket with no real evidence in it</td><td>An agent that can never say "unclear" is not safe to trust</td></tr>
</table>

<div class='concept-box'>
<strong>The three hours, in one line each</strong>
<table class='compare-table'>
<tr><td><strong>A1</strong></td><td>RAG could not remember, could not know today, and could not act. Tools fixed two of those — and the model began <em>choosing</em>.</td></tr>
<tr><td><strong>A2</strong></td><td>Memory made bounded and made to survive a restart; search let it see a world that moved on. A capability without context still failed invisibly.</td></tr>
<tr><td><strong>A3</strong></td><td>It can act, remember and look outward — so we put a human in front of the write, and proved the rejection path with a hash.</td></tr>
</table>
</div>

<div class='topic-header'>
<strong>&#128279; What we did not build today</strong><br><br>
<strong>Multi-agent orchestration</strong> &mdash; a supervisor routing work to specialists, with handoffs
and escalation. It is the natural next step once one gated agent is well understood, and it is mostly
an <em>economics</em> question: several small specialists versus one large generalist, measured in
tokens.<br><br>
<strong>Agent taxonomies</strong> &mdash; reflex, goal-based and routing agents. The one idea worth
carrying now: a cheap router that sends easy cases to a small model and hard ones to a large model
typically cuts cost by roughly half with no accuracy loss on the hard cases. Ask your teams whether
every request really needs the big model.
</div>

In [ ]:
# Housekeeping — restore the queue so the notebook re-runs from a clean state.
shutil.copy('../data/_a3_backup.json', TICKETS_FILE)
os.remove('../data/_a3_backup.json')
try:
    triage_ui.close()
except Exception:
    pass
print('Queue restored. Fingerprint:', fingerprint())
for t in load_tickets():
    print(f"  {t['id']}  {t['status']}")